#**Exercise 1 - Text representation with Tf-Idf and basic IR models [Solutions]**

This notebook contains solutions to the exercise session.

###**Exercise: Load your own data**

For this exercise, you will load a subset of the [Simple English Wikipedia](https://simple.wikipedia.org/wiki/Main_Page), which is small and easy to load.

Part 1: Load the dataset and implement a boolean model, a vector space model
and a probabilistic model on the dataset for the following queries:

1. Earth's atmosphere
2. Agricultural crops
3. Parts of the human body
4. What are the official languages of countries?
5. Best places to travel

Part 2: Rank the documents and compare the top 5 results for each model. What differences do you see? Try passing longer queries to the models, for example, "best places to travel as a tourist" instead of "best places to travel". Does that change the result?

Part 3: What happens to the results if you don't lemmatize or remove stop words? Can you think of any additional steps you can add to the preprocessing?

Part 4: Try a different similarity measure for the vector space model. Which one returns more suitable results?

In [ ]:
!pip install datasets
from datasets import load_dataset

In [ ]:
data = load_dataset("wikipedia", "20220301.simple")['train']
data = data.select(range(200))
#the text in the data can be accessed by data['text']
print(data)
print(data['text'][0])

###**Implementing a Boolean Model**

A boolean model takes in a query in boolean logic and returns the documents whose features match that query. Although simplistic, it is easy to implement, and can have varying applications. Here we look to [this codebase](https://github.com/mayank-02/boolean-retrieval-model/tree/main) for the implementation of a boolean IR model.

In [ ]:
# Let's get some of the imports out of the way

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

In [ ]:
# We clone the repository

!git clone https://github.com/mayank-02/boolean-retrieval-model.git
%cd boolean-retrieval-model/

In [ ]:
# The implementation requies data in the form of text files, so let's create them.
# We will use the data ids provided for each article in the dataset as filenames so we can retrieve them later

%mkdir data
for doc in data:
  with open (f'data/{doc["id"]}.txt', "w") as outfile:
    outfile.write(doc['text'])

In [ ]:
from BooleanModel import BooleanModel

In [ ]:
model = BooleanModel("./data/*") # pass the data to the model

In [ ]:
# define a query in the form of boolean logic

# query = ("earth & atmosphere")
# query = ("agricultural & crops")
# query = ("parts & human & body")
# query = ("country & official & language")
query = ("travel & best & tourism")

In [ ]:
# pass it to the model

results = model.query(query)
print(results)

In [ ]:
#let's look at the results

import re
regex = re.compile(r'\d+')

result_ids = []
for result in results:
  id = regex.findall(result)[0]  # get the data ids from the filenames in the results
  result_ids.append(id)

for article in data:
  if article["id"] in result_ids:  # use the ids to get those datapoints from the dataset object
    print(f"{article}\n")

This type of model can be fast and efficient, since it is just doing literal string matching. For applications where you just want to do a keyword search, for example, this is very good. Moreover, it lets you also define words that you do not want to appear in the documents, which is not possible in the vector space or probabilistic models.

On the other hand, you have to be able to fit your queries into the structure of boolean logic. This becomes difficult for longer, more complex queries. Moreover, this type of model does not work for queries that have a higher level of abstraction required.

###**Implementing a Vector Space Model**

A vector space model takes a vector representation of the document and the query and ranks documents according to a similarity or distance measure between the query and document vector.

We will use the tf-idf weights we calculated above for our vector space model. The query can be represented as a simple binary vector where 1 represents the presence of a word and 0 represents the absense. In this implementation, the query vector is basically a vector of the size of the vocabulary of the corpus, with a tfidf weight assigned to the words present in the query.


In [ ]:
# Let's get some of the imports out of the way
import nltk
import pandas as pd
import math
import numpy as np
import string
nltk.download('wordnet')

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [ ]:
# defining the preprocessing function

digits = re.compile(r'\d')
lemmatizer = WordNetLemmatizer()
def preprocess(doc):   #this takes in a string and converts it into a list
  doc = doc.split()
  preprocessed_text = []
  for text in doc:
    text = text.translate(str.maketrans('', '', string.punctuation)) #remove punctuation
    text = text.lower() #convert to lower case
    words = word_tokenize(text) #tokenize the text
    words = [word for word in words if word not in stopwords.words('english')] #remove stopwords
    words = [word for word in words if not digits.match(word)] # additional step : removing digits
    words = [lemmatizer.lemmatize(word) for word in words] #lemmatize
    if words != []:
      preprocessed_text.append(words[0])
  return preprocessed_text

In [ ]:
# create the tfidf matrix

from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(tokenizer=preprocess)
X = vectorizer.fit_transform(data['text'])  # this will take some time to run
tfidf_df = pd.DataFrame(X.toarray(), index=range(len(data)), columns=vectorizer.get_feature_names_out())
tfidf_df = tfidf_df.round(2)
tfidf_df

In [ ]:
# let's define the query again:

# query = "Earth's atmosphere"
# query = "Agricultural crops"
# query = "Parts of the human body"
# query = "What are the official languages of countries?"
query = "Best places to travel as a tourist"

# query = 'name parts of the human body.'


# we will have to preprocess the query before we convert it into a vector:

query = preprocess(query)
print(query)

In [ ]:
query_vector = vectorizer.transform([" ".join(query)])
print(query_vector)
print(query_vector.toarray()) # this is the actual vector

In [ ]:
# compute cosine similarities
from sklearn.metrics.pairwise import cosine_similarity

cosine_similarities = cosine_similarity(query_vector, X)
results = [(data[i], cosine_similarities[0][i]) for i in range(len(data))]
results.sort(key=lambda x: x[1], reverse=True)
for doc, similarity in results[:5]:
    print(f"Similarity: {similarity:.2f}\n{doc}\n")

In [ ]:
# compute euclidean distance
from sklearn.metrics.pairwise import euclidean_distances

euclidean_dist = euclidean_distances(query_vector, X)
results = [(data[i], euclidean_dist[0][i]) for i in range(len(data))]
results.sort(key=lambda x: x[1])
for doc, similarity in results[:5]:
    print(f"Distance: {similarity:.2f}\n{doc}\n")

Both similarity metrics return the same rankings.

###**Implementing Okapi BM25 (Probabilistic model)**

The Best-Match 25 algorithm follows the probabilistic retrieval framework, and uses term frequency and document length normalisation to determine the relevance of a document given a query. It operates with the underying assumption that a document generates a query.

The BM25 score for a document D with respect to a query Q is calculated as the sum of the scores for individual query terms. The formula for calculating the BM25 score is as follows:

$BM25(D, Q) = ∑(IDF(q) * ((TF(q, D) * (k1 + 1)) / (TF(q, D) + k1 * (1 — b + b * (|D| / avgdl)))))$

In this formula, IDF(q) represents the inverse document frequency of the query term q, TF(q, D) denotes the modified term frequency of term q in document D, |D| represents the length of document D, and avgdl is the average document length in the corpus. Parameters k1 and b are tunable constants that control the impact of term frequency saturation and document length normalization, respectively.

Note that BM25 measures TF and IDF differently. BM25 uses a modified term frequency that takes into account saturation effects to prevent overemphasizing heavily repeated terms. For IDF it assigns higher weights to terms that are rare in the corpus and lower weights to terms that are common.

Following is an implementation of the BM25 algorithm, taken from [this paper](http://www.cs.otago.ac.nz/homepages/andrew/papers/2014-2.pdf).

In [ ]:
!pip install rank_bm25

In [ ]:
from rank_bm25 import BM25Okapi

In [ ]:
preprocessed_text = [preprocess(doc) for doc in data['text']]

In [ ]:
bm25 = BM25Okapi(preprocessed_text)

In [ ]:
# query = "Earth's atmosphere"
# query = "Agricultural crops"
# query = "Parts of the human body"
# query = "What are the official languages of countries?"
query = "Best places to travel as a tourist"
# query = 'name parts of the human body.'

query = preprocess(query)
print(query)

In [ ]:
doc_scores = bm25.get_scores(query)

score_to_id = {}
for i in range(len(doc_scores)):
  score_to_id[str(i)] = float(doc_scores[i])

ranking = sorted(score_to_id.items(), key=lambda x:x[1], reverse=True)

for k,v in ranking[:5]:
  print(f'Document: {data[int(k)]}\n Score: {v}\n')

While the top 3 or 4 ranked documents are the same for both vector space and probabilistic models (they both use tfidf, although with different variations) the differences in the models come out when we go down the rankings. Although just based on word frequencies, tfidf can be a powerful representation that is able to identify relevance in a meaningful manner.

###**What happens to the tfidf matrix if you don't preprocess the text? How does that affect the results from your vector space model?**


In [ ]:
# define the preprocessing function again - now it just splits the documents into a list without any other preprocessing

def preprocess(doc):   #this takes in a string and converts it into a list
  doc = doc.split()
  # preprocessed_text = []
  # for text in doc:
  #   text = text.translate(str.maketrans('', '', string.punctuation)) #remove punctuation
  #   text = text.lower() #convert to lower case
  #   words = word_tokenize(text) #tokenize the text
  #   words = [word for word in words if word not in stopwords.words('english')] #remove stopwords
  #   words = [lemmatizer.lemmatize(word) for word in words] #lemmatize
  #   if words != []:
  #     preprocessed_text.append(words[0])
  return doc


In [ ]:
# create the matrix again

vectorizer = TfidfVectorizer(tokenizer=preprocess)
X = vectorizer.fit_transform(data['text'])
tfidf_df = pd.DataFrame(X.toarray(), index=range(len(data)), columns=vectorizer.get_feature_names_out())
tfidf_df = tfidf_df.round(2)
tfidf_df

You will observe that there is a lot of noise retained in the matrix that does not add any value to the representations. You are essentially storing noise and adding compute time, and assigning weights to words/characters that are irrelevant. You can try running the queries again to see how that impacts the type of documents your model will return.

In [ ]:
# let's define the query again:

# query = "Earth's atmosphere"
# query = "Agricultural crops"
# query = "Parts of the human body"
# query = "What are the official languages of countries?"
query = "Best places to travel as a tourist"
# query = 'name parts of the human body.'


query = preprocess(query)
print(query)

Notice that the query also retains the stop words - so you will be computing similarities for 'of' and 'the' that add no additional information to the meaning of your query - they just serve a grammatical purpose.

###**Vector Space Model**

In [ ]:
query_vector = vectorizer.transform([" ".join(query)])
print(query_vector)

In [ ]:
cosine_similarities = cosine_similarity(query_vector, X)
results = [(data[i], cosine_similarities[0][i]) for i in range(len(data))]
results.sort(key=lambda x: x[1], reverse=True)
for doc, similarity in results[:5]:
    print(f"Similarity: {similarity:.2f}\n{doc}\n")

In [ ]:
euclidean_dist = euclidean_distances(query_vector, X)
results = [(data[i], euclidean_dist[0][i]) for i in range(len(data))]
results.sort(key=lambda x: x[1])
for doc, similarity in results[:5]:
    print(f"Distance: {similarity:.2f}\n{doc}\n")

###**Probabilistic Model**

In [ ]:
preprocessed_text = [preprocess(doc) for doc in data['text']]

In [ ]:
bm25 = BM25Okapi(preprocessed_text)

In [ ]:
doc_scores = bm25.get_scores(query)

score_to_id = {}
for i in range(len(doc_scores)):
  score_to_id[str(i)] = float(doc_scores[i])

ranking = sorted(score_to_id.items(), key=lambda x:x[1], reverse=True)

for k,v in ranking[:5]:
  print(f'Document: {data[int(k)]}\n Score: {v}\n')